<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - Reverse Engineering</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 1: Introduction to Reverse Engineering</h1>

<hr style="border: 1px solid white"></hr>

<h1 style="font-size:30px; font-family:Monaco">1 - What is RE?</h1>

In a general sense, taking a program (that you don't have the source code for) and figuring out what the source code was.

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">1.1 - Execution of a program</h2>

+ How to tell a computer what to do
+ Compiled programs reduce to a *binary* or *executable*

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">1.2 - What is a binary?</h2>

Computers store instructions for what to do as binary numbers.\
For example, the assembly code:
>```asm
>        ADD RAX, 10 ; adds 10 to register A
>```
compiles to\
`01001000 10000011 11000000 00001010` or `48 83 c0 0a` in hexadecimal\
Each instruction tells the computer which operations to do and on what data to do it. Usually, there are two kinds of data that the CPU operates on: *immediates*, which are static numbers like 1, 5, or 0xDEADBEEF, passed in the instruction, and *registers*.

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">1.3 - What is a <strong>register</strong>?</h2>

The CPU needs somewhere to store data nearby, so it doesn't have to read and write between RAM often, which is a slow process. These places are called registers, and act like small buffers directly on the CPU. In x86-64, which is what Intel/AMD chips run on, the register layout is
+ `RAX`-`RDX`: The most prevalent registers in modern assembly Their most common/canonical uses are
    * `RAX`: **A**ccumulator in repeated operations or return value of functions
    * `RBX`: Memory addressing(the **b**ase) or as for local/callee-saved variables
    * `RCX`: **C**ounter for repeated operations and loops. The `LOOP` instruction expands to `SUB RCX, 1 ; JNZ`
    * `RDX`: Holds extra **d**ata when other registers are full or the value is too large to fit in a single register. Commonly used to extend `RAX`
+ `RSI` and `RDI`: The **s**ource and **d**estination **i**ndices for string operations
+ `RSP` and `RBP`: The **s**tack and **b**ase **p**ointers for stack frames
+ `R8`-`R15`: General purpose registers
+ `RIP`: The **i**nstruction **p**ointer; It tells the program where the current code that is being evaluated is located in memory
+ `RFLAGS`: The **flags** register. Stores information about operations, often used for checking comparisons and interruptions

So, we know how it holds data, but how does it use that data?

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">1.4 - Using Operations</h2>

Each operation tells the computer what to do. The general format in Intel syntax is `OP Rd, Rs`(OP=operation, Rd=destination register, Rs=source register). Another format is AT&T syntax, which is in the form `OP %Rs, %Rd`. However, AT&T syntax is evil and thus will not be dicussed for the remainder of the lecture. Some applications will default to AT&T so it is good to know what it looks like. Some common operations are
+ `MOV`: **Mov**es the value from Rs into Rd, copying the data
+ `ADD`: **Add**s the value in Rs to the value in Rd, storing it in Rd
+ `SUB`: **Sub**tracts the value in Rs from the value in Rd, storing it in Rd
+ `XOR`: **Xor**s the value in Rs with the value in Rd, storing it in Rd
+ `POP`: Reads and **pop**s the top of the stack into Rd, does not take an Rs
+ `JMP`: Make the program **j**u**mp** to a given address in Rd, does not take an Rs
+ `CMP`: **C**o**mp**ares Rd to Rs, stores no data except for flag results
+ `JZ`/`JE`: **J**umps if the **z**ero flag is set(**e**qual result from a comparison)
+ `JNZ`/`JNE`: Same as `JZ`/`JE`, but in the opposite case(**z**ero flag is **n**ot set, or the comparison result is **n**ot **e**qual)
+ `CALL`: Performs a function **call** to the given label, does not take Rd or Rs, arguments are passed in registers and the stack(see #3)
+ `SYSCALL`: Performs a **sys**tem **call**, the arguments are passed in registers and the stack, not in the instruction

Because x86 is a CISC(**c**omplex **i**nstruction **s**et **a**rchitecture) there are a lot of instructions, and you will probably not see most of them.

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">1.5 - Dereferencing Notation</h2>

A very common notation in x86 assembly is the `[]` notation. For example, `[rax]` means to read the value at the memory address in `RAX`. Additionally, you can use modifiers, like `[rbx-0x8]` would read the data from 8 bytes lower than the address at `RBX`.

<hr style="border: 1px solid white"></hr>

<h1 style="font-size:30px; font-family:Monaco">2 - Calling Convention</h1>

When making a function call in x86-64, there are policies/standards for the actual carrying out of the operations. Parameters are passed in the following order(on linux):
1. `RDI`
2. `RSI`
3. `RDX`
4. `RCX`
5. `R8`
6. `R9`
7. The stack

The caller must allocate stack space between the previous stack frame and the new stack frame, and when the function returns, return values are passed in `RAX`

Note: Other operating systems and ISAs use different calling conventions, for example, the x64 Microsoft calling convention can be seen [here](https://en.wikipedia.org/wiki/X86_calling_conventions#Microsoft_x64_calling_convention)

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">2.1 - What are the function prologue and epilogue?</h2>

The function prologue and epilogue are the preparation and cleanup sections of a function call. You don't want your functions overwriting variables, do you?
##### Prologue
The prologue controls the entrance into a function, making sure that collisions don't occur(functions sharing memory they shouldn't). They have various forms, but the typical form is(as produced by GCC):
>```asm
>        push ebp
>        mov ebp, esp
>        sub esp, N
>```
where N is the size of the function stack.

##### Epilogue
The epilogue controls the exit of a function, and ensures that the function cleans up after istelf. The standard format is
>```asm
>        leave
>        ret
>```
where `leave` is an alias for
>```asm
>        mov esp, ebp
>        pop ebp
>```
and `ret` is an alias for
>```asm
>        pop rip
>```

Additionally, the function perilogue (the prologue and epilogue together), often contains code preventing buffer overflows (a.k.a. BOFs or stack smashing), which complicates binary exploitation (you'll learn about this later).

<hr style="border: 1px solid white"></hr>

<h1 style="font-size:30px; font-family:Monaco">3 - Comprehending Binaries</h1>

So, we know how assembly works, but how does that help us when we have a binary? We use tooling to disassemble binary code back into assembly.

>```bash
>$ objdump -M intel -d file
>```

Once we have that we can look at the assembly and determine what the program does and how to break it.

<hr style="border: 1px solid white"></hr>

<h1 style="font-size:30px; font-family:Monaco">Challenge 1</h1>

Try seeing if you can figure out how `chal1` works using `objdump`. (Tip: try to find `main`)

Looks complicated, doesn't it? Lets break this down...

Assuming you ran the command correctly, you should see something that looks like(don't worry if it's not exact):\
<img src="./media/chal1.png">

Lets see what that code is actually doing:\
<img src="./media/chal1-annotated.png">

Notice how the function calls pass in values through `rdx`, `rsi`, and `rdi`?(`lea` is the 'address-of' operator for assembly, it's used here to get a pointer from the stack)\
Do you notice how after the `strtoull()` call, it moves a value from `rax` to the stack?(`[rbp-0x8]`) That's storing the return value as a variable!\
Remember: if you see `je`/`jz` or `jne`/`jnz`, it's often a conditional statement(`if`/`else`)

Still confused on jumps? `je`, as discussed is "jump if equal". When using the `cmp` operation, if the two values are equal, it sets the `zero` flag. This is because `cmp` is a shortcut to subtract the two values but doesn't store them(two equal values subtracted is 0). `jne` works the same way, but checks if the `zero` flag *isn't* set.

See if you can figure out what the program is doing now! Can you crack it? (hint: look at documentation if you don't know what a function does or how it works)

<hr style="border: 1px solid white"></hr>

<h1 style="font-size:30px; font-family:Monaco">Challenge 2</h1>

This one's more complicated than the last. If you ran `objdump` correctly, you should see seomthing like:\
<img src="./media/chal2.png">

Try to solve this one without looking at the following breakdown guide! See how much you can understand just by reading it.\
**Tip**: if you want to see the arrows for jumps, pass the `--visualize-jumps` flag to `objdump`

Still stuck? Let's look at the breakdown of this code.\
<img src="./media/chal2-annotated.png">

Broken down, it seems pretty similar to a regular program, right?\
There are some confusing parts, especially the <span style="color: #ff7000">stack canary</span> and the <span style="color: #00ff00">loop</span>. Lets dig into those.

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">Stack Canary</h2>

The `fs` register is the 'thread-specific' register. It holds information about the current thread, in this case, the random number used as the overflow protection cookie.\
The set block moves the canary into `rax`, stores that in a local variable(`[rbp-0x8]`), then xors `eax` with itself to scrub it from the register.\
The check block loads the value back into `rdx`, compares it to `fs:0x28` using a `sub` operation, and calls the `__stack_chk_fail` function if they don't match. This tells the program that stack smashing occurred.

<hr style="border: 1px solid white"></hr>

<h2 style="font-size:20px; font-family:Monaco">Loops</h2>

This example of a loop in x86 is heavily overcomplicated. This is primarily due to being compiled without optimizations. It's doing extra work of reading and writing every value when it could be using registers to avoid much of the semantics.\
In actuality, this loop could be completed with only a single arithmetic operation. Thanks to SIMD(**s**ingle **i**nstruction **m**ultiple **d**ata) the computer can perform all of the additions at the same time, using an operation like `PADDB`(**p**arallel **add** **b**ytes), but that's a little out of scope.\
What you need to really know for loops is to identify where a jump goes backward to previous instructions, and identify what changes occur.

<hr style="border: 1px solid white"></hr>

Now, knowing this, crack that program!